# Playtime, Content Tags, and Community Reception of Visual Novels

## Motivation

I enjoy reading visual novels and am interested in how their length
and subject matter relate to readers' evaluations. Some visual novels
require many hours to complete, while others offer much shorter
experiences. I want to investigate whether longer works tend to
receive higher scores and whether this relationship differs across
content tags.

## Research Questions

1. What is the relationship between visual novel playtime and average user scores?

2. How do average user scores and vote counts differ across selected content tags?

3. Does the relationship between playtime and average user scores differ across these content tags?

## Data Source

I plan to use the VNDB API:
https://api.vndb.org/kana

Each row in my dataset represents one visual novel. I plan to focus
on visual novels originally written in Japanese.

## Potential Variables

- `id`: The unique VNDB identifier for each visual novel.
- `title`: The title of the visual novel.
- `released`: The release date.
- `length_minutes`: The average user-submitted playtime in minutes.
- `length_votes`: The number of submitted playtime estimates.
- `average`: The raw average of user scores.
- `rating`: VNDB's Bayesian rating, for an additional comparison.
- `votecount`: The number of rating votes.
- `tags`: The recorded tags associated with each visual novel.

## Initial Expectations

[What relationships do I expect to find, and why?
These are hypotheses, not conclusions.]

## Limitations

VNDB scores and votes reflect participation within the VNDB community,
rather than all readers or total sales. Playtime estimates are
user-submitted and may be based on few responses.

Tags can overlap, and missing tag records do not necessarily mean that
a work lacks a particular theme. Release year and other characteristics
may also be related to both playtime and scores.

This study will examine associations and will not establish that longer
playtime causes higher scores.

## Decisions Still to Make

- [Which language or languages should I focus on?]
- [Which release years should I include?]
- [How should I handle missing scores and works with very few votes?]

In [23]:
import requests
import pandas as pd

query = {
    "filters": ["olang", "=", "ja"],
    "fields": (
    "title,alttitle,released,languages,average,rating,votecount,"
    "length_minutes,length_votes,"
    "tags{id,name,category,rating,spoiler,lie}"),
    "results": 100,
    "sort": "id",
    "page": 1
}

response = requests.post(
    "https://api.vndb.org/kana/vn",
    json=query,
    timeout=30
)

response.raise_for_status()
payload = response.json()

data = pd.DataFrame(payload["results"])
data

,alttitle,average,id,languages,length_minutes,length_votes,rating,released,tags,title,votecount
0,みんなでニャンニャン,57.8,v1,"[en, ja]",900.0,3,57.8,2003-10-17,"[{'category': 'tech', 'id': 'g32', 'lie': Fals...",Minna de Nyan Nyan,691
1,加奈～いもうと～,74.2,v2,"[en, ja, ko, ru, vi, zh-Hans]",627.0,55,74.2,1999-06-25,"[{'category': 'cont', 'id': 'g8', 'lie': False...",Kana ~Imouto~,2047
2,うたわれるもの,78.2,v3,"[en, es, ja, ko, ru, zh-Hans, zh-Hant]",2205.0,287,78.2,2002-04-26,"[{'category': 'cont', 'id': 'g4', 'lie': False...",Utawarerumono,6105
3,None,87.3,v4,"[en, es, fr, it, ja, ko, pt-br, ru, vi, zh-Hans]",4680.0,497,87.3,2004-04-28,"[{'category': 'tech', 'id': 'g32', 'lie': Fals...",CLANNAD,11541
4,リトルバスターズ！,86.4,v5,"[en, es, ja, ko, pt-br, ru, zh-Hans]",4968.0,331,86.4,2007-07-27,"[{'category': 'cont', 'id': 'g2', 'lie': False...",Little Busters!,9343
...,...,...,...,...,...,...,...,...,...,...,...
95,沙耶の唄,78.6,v97,"[de, en, es, fr, it, ja, ko, pl, pt-br, ru, th...",375.0,1223,78.6,2003-12-26,"[{'category': 'cont', 'id': 'g8', 'lie': False...",Saya no Uta,24448
96,アカイイト,80.0,v98,"[en, ja, ko, zh-Hans, zh-Hant]",1800.0,21,80.0,2004-10-21,"[{'category': 'cont', 'id': 'g5', 'lie': False...",Akai Ito,297
97,水夏~SUIKA~,68.9,v99,"[en, ja, zh-Hans]",1364.0,8,68.9,2001-07-27,"[{'category': 'cont', 'id': 'g8', 'lie': False...",Suika,588
98,もしも明日が晴れならば,75.2,v100,"[ja, ko, zh-Hans]",1388.0,6,75.2,2006-02-24,"[{'category': 'ero', 'id': 'g23', 'lie': False...",Moshimo Ashita ga Hare Naraba,152


In [24]:
data[
    ["title", "length_minutes", "length_votes", "average", "votecount"]
]

,title,length_minutes,length_votes,average,votecount
0,Minna de Nyan Nyan,900.0,3,57.8,691
1,Kana ~Imouto~,627.0,55,74.2,2047
2,Utawarerumono,2205.0,287,78.2,6105
3,CLANNAD,4680.0,497,87.3,11541
4,Little Busters!,4968.0,331,86.4,9343
...,...,...,...,...,...
95,Saya no Uta,375.0,1223,78.6,24448
96,Akai Ito,1800.0,21,80.0,297
97,Suika,1364.0,8,68.9,588
98,Moshimo Ashita ga Hare Naraba,1388.0,6,75.2,152


In [25]:
print(len(data))

data[["length_minutes", "average"]].isnull().sum()

100


length_minutes    4
average           0
dtype: int64

In [26]:
mask = data["length_minutes"].isnull()

data[mask][["title", "length_minutes", "length_votes", "votecount"]]

,title,length_minutes,length_votes,votecount
64,Mai-HiME - Unmei no Keitouju,NaN,0,40
76,Spitan Spirits Expedition -in the Phantasmagoria-,NaN,0,16
79,Tsuki wa Higashi ni Hi wa Nishi ni ~Operation ...,NaN,0,64
87,_summer,NaN,0,42


In [27]:
for tag in data.loc[0, "tags"]:
    if tag["category"] == "cont":
        print(tag["name"])

Comedy
Maid Heroine
Protagonist's Senpai as a Heroine
Miko Heroine
Robot Heroine
Male Protagonist
Deredere Heroine
Protagonist's Childhood Friend as a Heroine
Alternate Dimensions
Cats
Catgirl Heroine
Trap Support Character
Under the Same Roof
Butler Support Character
Kemonomimi Support Character
Doggirl Heroine
Rabbitgirl Heroine
Childishly Violent Heroine
Perverted Protagonist
Thief Heroine
Clumsy Heroine
Kemonomimi
Waitress Heroine
Heroine with Ahoge
University Student Protagonist
University Student Heroine
Heroine with Tan Lines
Waiting Staff Protagonist
Landlady Heroine
Heroine with Armpit Hair
Heroine from a Different World


## Earlier API Practice: Language Availability

In [28]:
def has_english(data: list):
    """
    Returns whether the language list contains English.
    """
    df_en = "en" in data
    return df_en

In [29]:
def has_chinese(data: list):
    """
    Returns whether the language list contains Chinese.
    """
    df_chs = ("zh-Hans" in data) or ("zh-Hant" in data)
    return df_chs

assert has_chinese(["ja", "zh-Hans"]) is True
assert has_chinese(["ja", "zh-Hant"]) is True
assert has_chinese(["ja", "en"]) is False

In [30]:
data["has_english"] = data["languages"].apply(has_english)

data[["title", "languages", "has_english"]]

,title,languages,has_english
0,Minna de Nyan Nyan,"[en, ja]",True
1,Kana ~Imouto~,"[en, ja, ko, ru, vi, zh-Hans]",True
2,Utawarerumono,"[en, es, ja, ko, ru, zh-Hans, zh-Hant]",True
3,CLANNAD,"[en, es, fr, it, ja, ko, pt-br, ru, vi, zh-Hans]",True
4,Little Busters!,"[en, es, ja, ko, pt-br, ru, zh-Hans]",True
...,...,...,...
95,Saya no Uta,"[de, en, es, fr, it, ja, ko, pl, pt-br, ru, th...",True
96,Akai Ito,"[en, ja, ko, zh-Hans, zh-Hant]",True
97,Suika,"[en, ja, zh-Hans]",True
98,Moshimo Ashita ga Hare Naraba,"[ja, ko, zh-Hans]",False


In [31]:
data["has_chinese"] = data["languages"].apply(has_chinese)

data[["title", "languages", "has_chinese"]]

,title,languages,has_chinese
0,Minna de Nyan Nyan,"[en, ja]",False
1,Kana ~Imouto~,"[en, ja, ko, ru, vi, zh-Hans]",True
2,Utawarerumono,"[en, es, ja, ko, ru, zh-Hans, zh-Hant]",True
3,CLANNAD,"[en, es, fr, it, ja, ko, pt-br, ru, vi, zh-Hans]",True
4,Little Busters!,"[en, es, ja, ko, pt-br, ru, zh-Hans]",True
...,...,...,...
95,Saya no Uta,"[de, en, es, fr, it, ja, ko, pl, pt-br, ru, th...",True
96,Akai Ito,"[en, ja, ko, zh-Hans, zh-Hant]",True
97,Suika,"[en, ja, zh-Hans]",True
98,Moshimo Ashita ga Hare Naraba,"[ja, ko, zh-Hans]",True
